# I2SB regressors: five architectures, one bridge, two cohorts

All five runs are `task: i2sb` on **identical data** -- `img_median_mad`, `scales: [3,3,3,3]`,
`x0 = T1ce`, `x1 = T1` -- with the **same** schedule (`kind: "i2sb"`, `beta_max: 0.3`), the same
`ddpm` posterior and the same `data_range: 2.0`. Only the regressor `r(x_t, t)` differs, so this
is an architecture comparison rather than one confounded by preprocessing.

| | regressor | prior | sees |
|---|---|---|---|
| **UNet (all)** | ADM UNet, step-embedded | -- | x_t + FLAIR, T1, T2 |
| **UNet (T1)** | ADM UNet, step-embedded | -- | x_t only |
| **CDLNet (T1)** | unrolled ISTA, sigma-adaptive threshold | l1 | x_t only |
| **SBCDLNet** | two-fidelity unrolled ISTA | l1 | x_t + FLAIR, T1, T2 |
| **SBGroupCDL** | two-fidelity unrolled ISTA | nonlocal group l1 | x_t + FLAIR, T1, T2 |

## Two cohorts, because the failure modes are opposite

Averaging over a random batch hides the thing you actually want to know. BraTS slices split into
two regimes that ask contradictory things of a synthesizer:

* **with tumour** -- does it *produce* the enhancement? Scored by **ET-PSNR**. Whole-brain PSNR
  barely moves on ~0.3% of voxels, so it cannot answer this.
* **without tumour** -- does it *refrain*? A net trained to add enhancement can add it where
  there is none. There is no ET region to score, so the headline is whole-brain PSNR plus
  **`p99.9+`**, the 99.9th percentile of the positive part of `recon - T1ce` over the brain: the
  brightest thing the method invented. Lower is better; the prior-only floor is the reference.

A method that wins on one cohort and loses on the other is the normal outcome, and averaging the
two would have shown neither.

"Without tumour" means **no enhancing tumour on this slice**, not a healthy subject -- BraTS is
all tumour patients, and `et_mask: true` drops subjects with no segmentation entirely, so both
cohorts come from the same subjects.

## Reading the conditioning column first

Three of these see all three contrasts and two see only the bridge state, so the five runs are
really two groups. SBCDLNet and SBGroupCDL cannot be made unconditional -- they need `x_1` as a
conditioning channel by construction (`C >= 2`), which is how their prior-fidelity term is
defined -- so a like-for-like unrolled-vs-unrolled comparison against CDLNet does not exist among
these five. Section 1 prints the input set per method so this stays visible.

Diagnostics: **one-shot** (`x_t = x_1`, one network call -- quality as a plain feed-forward
synthesizer), **full recon** at fixed `NFE` (what you ship), and an **NFE sweep** (whether the
bridge is doing anything). `full - one-shot` is what the sampling bought.

In [ ]:
import os, sys, json, gc, time
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

import datasets                                  # registers the loaders
from datasets.registry import build_loader
from models import build_model
from training.common import load_model
from sb.base import build_schedule, n_steps, forward_std, predict_x0
from sb.i2sb import i2sb_sample

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
npy = lambda t: t.detach().cpu().numpy()

# ---- the runs to compare: (label, saved config.json) ------------------------------------
# Missing runs are dropped with a warning rather than raising, so this works while some are
# still training.
RUNS = [
    ("UNet (all)",   "trained_nets/brats/I2SB_Unet_T1ce_from_all/config.json"),
    ("UNet (T1)",    "trained_nets/brats/I2SB_Unet_T1ce_from_T1/config.json"),
    ("CDLNet (T1)",  "trained_nets/brats/I2SB_CDLNet_T1ce_from_T1/config.json"),
    ("SBCDLNet",     "trained_nets/brats/I2SB_SBCDLNet_T1ce_from_T1_medmad/config.json"),
    ("SBGroupCDL",   "trained_nets/brats/I2SB_SBGroupCDL_T1ce_from_T1_medmad/config.json"),
]

# ---- evaluation knobs -------------------------------------------------------------------
SPLIT     = "val"
NFE       = 20      # sampling budget every method is scored at -- FIXED, so the headline
                    # comparison is at equal cost
CROP      = 192     # fixed CENTER crop, so every method sees identical pixels. Divisible by
                    # CDLNet's stride 2 and the UNet's 8. Training used random 128 crops and val
                    # is native 240; 192 is the repo's usual testbench size. SBUnet's
                    # `image_size` only picks which LEVELS get attention at build time, so
                    # evaluating at a different size is fine (see models/sb_unet.py).
SEED      = 0

# ---- how many subjects to look at -------------------------------------------------------
PANEL_N   = 4       # subjects shown per image cell. Re-run that cell to advance to the next
                    # PANEL_N; call reshuffle() for a fresh random order. See section 8.

# ---- cohort definition ------------------------------------------------------------------
N_PER_COHORT   = 16   # slices EVALUATED per cohort. This is the pool the image cells page
                      # through, so it caps how many subjects you can ever see -- a recon you
                      # did not compute cannot be displayed. It also sets the sample size for
                      # every number in section 5. Cost is one sampler pass over
                      # N_PER_COHORT slices per method per cohort, so raising it is cheap
                      # relative to re-loading the nets; raise it before you raise NFE_GRID.
TUMOUR_MIN_PX  = 100  # >= this many enhancing-tumour voxels -> "with tumour". ~100 px at 192^2
                      # is ~0.27% of the crop, i.e. a lesion you can actually see.
CLEAR_MAX_PX   = 0    # <= this many -> "without". 0 = strictly none. Slices between the two
                      # thresholds are DROPPED, so neither cohort is contaminated by the
                      # ambiguous middle.
SCAN_BATCH     = 16   # loader batch while scanning for cohort members
MAX_SCAN       = 40   # give up after this many batches

DO_NFE_SWEEP = True                      # section 7; costs len(NFE_GRID) samplers per method
NFE_GRID     = [1, 2, 5, 10, 20, 50]
NFE_SWEEP_N  = 8                         # slices used for the SWEEP only (None = the whole
                                         # cohort). The sweep is an aggregate curve and does not
                                         # need the full pool, so this keeps a wide N_PER_COHORT
                                         # from making sum(NFE_GRID) network passes expensive.

# ---- display window (fixed, not per-image) ----
# Panels are shown on [VMIN, VMAX] rather than per-image percentiles. med/MAD + scales=3 puts the
# brain in roughly [-1, 1], and a fixed window is the only way five methods are comparable:
# per-image windowing renormalizes each panel to its own contents, so a method that
# systematically under-predicts enhancement gets stretched back to full range and looks fine.
VMIN, VMAX = -1.0, 1.0

print(f"repo: {REPO_ROOT}\ndevice: {device}")

## 1. Load the configs, and audit what each method actually sees

The comparison is only meaningful if the data blocks and the bridge agree. Anything that differs
is printed as a `[WARN]` -- `image_key`, `scales`, `x0_idx`/`x1_idx`, `root`, and the schedule all
silently change what the numbers mean.

`inputs` is the union of the bridge prior and the conditioning contrasts, i.e. every contrast the
network can use. Methods sharing an `inputs` set are directly comparable; methods that do not are
not, however similar their PSNR looks.

In [ ]:
STORED = ["FLAIR", "T1", "T1ce", "T2"]        # cmap_config.contrasts order
NA = f"{'-':>10}"

runs = []
for label, p in RUNS:
    if not os.path.exists(p):
        print(f"[skip] {label}: no config at {p}")
        continue
    with open(p) as f:
        cfg = json.load(f)
    if not os.path.exists(os.path.join(os.path.dirname(p), "net.ckpt")):
        print(f"[skip] {label}: config but no net.ckpt in {os.path.dirname(p)}")
        continue
    runs.append({"label": label, "cfg_path": p, "cfg": cfg, "dir": os.path.dirname(p)})
if not runs:
    raise RuntimeError("no finished runs found -- check the paths in RUNS")

base_data = dict(runs[0]["cfg"]["data"][SPLIT])

for field in ("x0_idx", "x1_idx", "x1_source", "scales", "image_key", "root"):
    vals = {r["label"]: str(r["cfg"]["data"][SPLIT].get(field)) for r in runs}
    if len(set(vals.values())) > 1:
        print(f"[WARN] runs disagree on data.{field}: {vals}")
for field in ("kind", "beta_max", "n_points", "posterior", "deterministic"):
    vals = {r["label"]: str(r["cfg"]["i2sb"].get(field)) for r in runs}
    if len(set(vals.values())) > 1:
        print(f"[WARN] runs disagree on i2sb.{field}: {vals} -- not the same bridge")

DATA_RANGE = float(runs[0]["cfg"].get("training", {}).get("data_range", 1.0))

hdr = f"{'method':<14} {'model':<11} {'C':>2} {'cond':<16} {'inputs':<18} {'schedule':<15} {'params':>11}"
print("\n" + hdr); print("-" * len(hdr))
for r in runs:
    dv, mc, ic = r["cfg"]["data"][SPLIT], r["cfg"]["model"], r["cfg"]["i2sb"]
    ci = list(dv.get("cond_idx", []))
    r["cond_idx"] = ci
    cond_names = [STORED[c] for c in ci]
    r["inputs"] = sorted({STORED[int(dv["x1_idx"])]} | set(cond_names))
    # Count from the ARCHITECTURE, not the checkpoint: net.ckpt also holds Adam's two moment
    # buffers, so torch.load'ing it costs ~3x the model just to sum shapes. The saved config has
    # `init` disabled, so building here is cheap (no power method) and touches no disk.
    try:
        _m = build_model(r["cfg"])
        r["params"] = sum(p.numel() for p in _m.parameters())
        del _m
    except Exception as e:
        r["params"] = -1
        print(f"[warn] {r['label']}: could not build to count params "
              f"({type(e).__name__}: {str(e)[:60]})")
    sched_s = f"{ic.get('kind','')}/beta={ic.get('beta_max')}"
    print(f"{r['label']:<14} {mc['type']:<11} {mc['params'].get('C','?'):>2} "
          f"{','.join(cond_names) or '-':<16} {','.join(r['inputs']):<18} {sched_s:<15}"
          + (f"{r['params']:>11,}" if r["params"] >= 0 else NA))

groups = {}
for r in runs:
    groups.setdefault(tuple(r["inputs"]), []).append(r["label"])
print(f"\ndata_range = {DATA_RANGE}  |  image_key = {base_data.get('image_key')}  "
      f"|  scales = {base_data.get('scales')}")
print("comparable groups (same inputs):")
for k, v in groups.items():
    print(f"   {', '.join(k):<20} -> {v}")
if len(groups) > 1:
    print("   ^ methods in DIFFERENT groups do not see the same contrasts. Compare within a\n"
          "     group first; across groups, any gap is partly an information gap.")

## 2. Build the two cohorts

Scans the split and bins slices by enhancing-tumour area into **with tumour**
(`>= TUMOUR_MIN_PX`) and **without** (`<= CLEAR_MAX_PX`), dropping the ambiguous middle so
neither cohort is contaminated. Same fixed centre crop and seed for both, so every method sees
identical pixels.

The conditioning stack is read with `cond_idx = [0, 1, 3]` so all three contrasts are available
regardless of what any individual run trained on; each method's own `cond` is rebuilt from that at
its own width.

In [ ]:
d = dict(base_data)
d.update(name="i2sb", cond_idx=[0, 1, 3], center_crop=CROP, random_flips=False,
         num_workers=0, batch_size=SCAN_BATCH, et_mask=True)
d.pop("crop_size", None)                     # center_crop instead -> deterministic

torch.manual_seed(SEED); np.random.seed(SEED)
try:
    loader = build_loader(d, shuffle=True, drop_last=False)
    it = iter(loader)
    _probe = next(it)
    if len(_probe) < 5:
        raise RuntimeError("loader returned no ET mask")
except (KeyError, RuntimeError) as e:
    raise RuntimeError(
        f"the cohort split needs the enhancing-tumour mask, but et_mask failed "
        f"({type(e).__name__}: {str(e)[:90]}). Backfill it with:\n"
        f"  python preprocessing/cmap.py --config config/BraTS/cmap.yaml --add-seg-only")

picked = {"tumour": [], "clear": []}
scanned = dropped = 0
torch.manual_seed(SEED); np.random.seed(SEED)
it = iter(build_loader(d, shuffle=True, drop_last=False))
for _ in range(MAX_SCAN):
    try:
        b = next(it)
    except StopIteration:
        break
    px = b[4].flatten(1).sum(1)
    for i in range(b[0].shape[0]):
        scanned += 1
        n = int(px[i])
        key = ("tumour" if n >= TUMOUR_MIN_PX else
               "clear" if n <= CLEAR_MAX_PX else None)
        if key is None:
            dropped += 1
        elif len(picked[key]) < N_PER_COHORT:
            picked[key].append(tuple(t[i:i + 1] for t in b[:5]) + (n,))
    if all(len(v) >= N_PER_COHORT for v in picked.values()):
        break

COHORTS = {}
for key, items in picked.items():
    if not items:
        print(f"[WARN] cohort '{key}' is EMPTY after scanning {scanned} slices -- "
              f"loosen TUMOUR_MIN_PX / CLEAR_MAX_PX or raise MAX_SCAN")
        continue
    cat = lambda j: torch.cat([it_[j] for it_ in items], dim=0).to(device)
    dc = cat(2)
    COHORTS[key] = {"x0": cat(0), "x1": cat(1), "mask": cat(3), "et": cat(4),
                    "FLAIR": dc[:, 0:1], "T1": dc[:, 1:2], "T2": dc[:, 2:3],
                    "et_px": [it_[5] for it_ in items]}
if len(COHORTS) < 2:
    print("[WARN] only one cohort was filled; the split comparison below will be partial")

COHORT_ORDER = [k for k in ("tumour", "clear") if k in COHORTS]
NICE = {"tumour": "with tumour", "clear": "without tumour"}


def stored_ch(co):
    """stored-channel index -> that cohort's tensor, for rebuilding any run's cond_idx."""
    return {0: co["FLAIR"], 1: co["T1"], 2: co["x0"], 3: co["T2"]}


def sub_cohort(co, n=None):
    """The first `n` slices of a cohort, as a cohort dict. Used for the NFE sweep, which does not
    need the full display pool."""
    if n is None or n >= co["x0"].shape[0]:
        return co
    out = {k: (v[:n] if torch.is_tensor(v) else v[:n]) for k, v in co.items()}
    return out


def cond_for(r, co):
    """That run's conditioning stack, at its own width and channel order. None when it trained
    unconditionally (cond_idx=[]), which is what predict_x0 expects."""
    ch = stored_ch(co)
    return torch.cat([ch[i] for i in r["cond_idx"]], dim=1) if r["cond_idx"] else None


print(f"scanned {scanned} slices, dropped {dropped} in the ambiguous middle "
      f"({CLEAR_MAX_PX} < ET < {TUMOUR_MIN_PX} px)")
for k in COHORT_ORDER:
    co, px_ = COHORTS[k], COHORTS[k]["et_px"]
    frac = float(co["et"].sum() / co["mask"].sum().clamp(min=1))
    print(f"  {NICE[k]:<16} n={co['x0'].shape[0]:<3} ET px per slice: "
          f"min={min(px_)} med={int(np.median(px_))} max={max(px_)}   "
          f"ET = {frac:.3%} of brain")

## 3. Metrics

PSNR and SSIM use the config's own `data_range` (2.0 for med/MAD at `scales=3`), so the numbers
line up with what `train_i2sb` logged rather than being offset by `20*log10(data_range)`.

`p99.9+` is the cohort-specific one: the 99.9th percentile of the **positive** part of
`recon - T1ce` over the brain. On a tumour-free slice the ground truth has no enhancement, so any
bright positive excursion is invented signal -- this is the hallucination measure, in med/MAD
units, lower is better.

In [ ]:
from training.metrics import ssim as ssim_fn


def region_psnr(gt, pred, region):
    """Area-normalized PSNR inside a binary region, on the config's data_range."""
    n = region.sum()
    if float(n) == 0:
        return float("nan")
    mse = float((region * (gt - pred) ** 2).sum() / n)
    return 10.0 * np.log10(DATA_RANGE ** 2 / max(mse, 1e-12))


def pos_p999(gt, pred, mask):
    """99.9th percentile of the POSITIVE part of (pred - gt) inside the brain: the brightest
    signal the method added that the ground truth does not have."""
    v = (pred - gt)[mask > 0.5].clamp(min=0).flatten().float()
    if v.numel() == 0:
        return float("nan")
    return float(torch.quantile(v, 0.999))


def score(pred, co):
    return {"psnr": region_psnr(co["x0"], pred, co["mask"]),
            "et_psnr": region_psnr(co["x0"], pred, co["et"]),
            "ssim": float(ssim_fn(co["x0"] * co["mask"], pred * co["mask"],
                                  data_range=DATA_RANGE).mean()),
            "p999": pos_p999(co["x0"], pred, co["mask"])}


PRIOR = {k: score(COHORTS[k]["x1"], COHORTS[k]) for k in COHORT_ORDER}
# The NFE sweep is scored on a subset, so its prior line must be too -- otherwise the dashed
# reference in section 7 belongs to a different set of slices than the curves.
PRIOR_SWEEP = {k: score(sub_cohort(COHORTS[k], NFE_SWEEP_N)["x1"],
                        sub_cohort(COHORTS[k], NFE_SWEEP_N)) for k in COHORT_ORDER}
print(f"data_range = {DATA_RANGE}\n")
print("prior only (x_hat = x1) -- the do-nothing floor every method must beat:")
for k in COHORT_ORDER:
    s = PRIOR[k]
    et = f"{s['et_psnr']:.3f}" if np.isfinite(s["et_psnr"]) else "  -  "
    print(f"  {NICE[k]:<16} psnr={s['psnr']:7.3f}  et_psnr={et:>7}  "
          f"ssim={s['ssim']:.4f}  p99.9+={s['p999']:.4f}")

## 4. Run every method on both cohorts

One method at a time -- a UNet and three unrolled nets will not sit in memory together, so each is
loaded, scored on **both** cohorts, and released. Each run brings its own schedule rebuilt from
its own `cfg["i2sb"]`, and nets carrying an internal copy get `assert_schedule_matches` called
against it: a silent mismatch there looks up the wrong bridge coefficients at every step.

In [ ]:
@torch.no_grad()
def evaluate(r, nfe_grid=()):
    ic = r["cfg"]["i2sb"]
    sched = build_schedule(kind=ic.get("kind", "brownian"), tau=ic.get("tau", 0.19),
                           n_points=ic.get("n_points", 1000),
                           beta_max=ic.get("beta_max", 0.3), device=device)
    net = load_model(r["cfg_path"], device=device)
    net.eval()
    if getattr(net, "attn_backend", None) == "flex":
        net.compile_flex()                        # fused kernel; compiles on the first call
    if hasattr(net, "assert_schedule_matches"):
        net.assert_schedule_matches(sched)        # model.params vs cfg["i2sb"] drift

    tc = ic.get("target_channels", 1)
    det, post = ic.get("deterministic", False), ic.get("posterior", "ddpm")
    clip = ic.get("clip_denoise", False)
    n = n_steps(sched)
    out = {}

    for key in COHORT_ORDER:
        co = COHORTS[key]
        c = cond_for(r, co)
        bs = co["x0"].shape[0]

        # one-shot: x_t = x1 at the t=1 end, ONE network call
        step_end = torch.full((bs,), n - 1, device=device, dtype=torch.long)
        sig_end = forward_std(sched, step_end, xdim=co["x0"].shape[1:])
        oneshot = predict_x0(net, co["x1"], sig_end, cond=c, target_channels=tc)

        def sample(nfe):
            torch.manual_seed(SEED)               # same bridge noise for every method
            if device.type == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            rec, _, _ = i2sb_sample(net, co["x1"], sched, cond=c, nfe=nfe, deterministic=det,
                                    posterior=post, clip_denoise=clip, target_channels=tc,
                                    log_count=1, verbose=False)
            if device.type == "cuda":
                torch.cuda.synchronize()
            return rec, time.time() - t0

        recon, secs = sample(NFE)
        out[key] = {"oneshot": oneshot, "recon": recon, "secs": secs, "nfe_curve": {}}

        if nfe_grid:
            # The sweep runs on a SUBSET: it is an aggregate curve, so it does not need the full
            # display pool, and sum(NFE_GRID) is ~4x the main NFE on its own.
            sub = sub_cohort(co, NFE_SWEEP_N)
            csub = cond_for(r, sub)

            def sample_sub(nfe):
                torch.manual_seed(SEED)
                rec, _, _ = i2sb_sample(net, sub["x1"], sched, cond=csub, nfe=nfe,
                                        deterministic=det, posterior=post, clip_denoise=clip,
                                        target_channels=tc, log_count=1, verbose=False)
                return rec

            out[key]["nfe_curve"] = {k: score(sample_sub(k), sub) for k in nfe_grid}

    del net
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return out


results = {}
for r in runs:
    print(f"-- {r['label']} ...", flush=True)
    results[r["label"]] = evaluate(r, NFE_GRID if DO_NFE_SWEEP else ())
print("done")

## 5. Summary, per cohort

`gain` is full-reconstruction minus one-shot: **what the reverse sampling bought.** Negative means
the sampler is hurting that architecture.

* **with tumour** -- read `ET` and `ET vs prior`. The latter is how much enhancement the method
  actually synthesized over answering T1 and doing nothing.
* **without tumour** -- read `p99.9+` against the prior's. A method scoring much higher than the
  prior is inventing enhancement on slices that have none.

In [ ]:
rows = {}
for key in COHORT_ORDER:
    co = COHORTS[key]
    rs = []
    for r in runs:
        o = results[r["label"]][key]
        s1, s2 = score(o["oneshot"], co), score(o["recon"], co)
        rs.append(dict(label=r["label"], inputs=",".join(r["inputs"]), params=r["params"],
                       one=s1["psnr"], full=s2["psnr"], gain=s2["psnr"] - s1["psnr"],
                       one_et=s1["et_psnr"], full_et=s2["et_psnr"],
                       et_vs_prior=s2["et_psnr"] - PRIOR[key]["et_psnr"],
                       psnr_vs_prior=s2["psnr"] - PRIOR[key]["psnr"],
                       ssim=s2["ssim"], p999=s2["p999"], secs=o["secs"]))
    rows[key] = rs

f = lambda v, w=7, p=3: (f"{v:>{w}.{p}f}" if np.isfinite(v) else f"{'-':>{w}}")

for key in COHORT_ORDER:
    co = COHORTS[key]
    print(f"\n### {NICE[key].upper()}   (n={co['x0'].shape[0]} slices, "
          f"ET = {float(co['et'].sum() / co['mask'].sum().clamp(min=1)):.3%} of brain)")
    if key == "tumour":
        hd = (f"{'method':<14} {'1-shot':>7} {'full':>7} {'gain':>7} | {'1shotET':>8} "
              f"{'fullET':>7} {'ETvsPrior':>10} | {'ssim':>6} {'s':>5}")
    else:
        hd = (f"{'method':<14} {'1-shot':>7} {'full':>7} {'gain':>7} | {'vsPrior':>8} "
              f"{'p99.9+':>8} {'vsPrior':>8} | {'ssim':>6} {'s':>5}")
    print(hd); print("-" * len(hd))
    for d in rows[key]:
        if key == "tumour":
            print(f"{d['label']:<14} {f(d['one'])} {f(d['full'])} {f(d['gain'])} | "
                  f"{f(d['one_et'], 8)} {f(d['full_et'])} {f(d['et_vs_prior'], 10)} | "
                  f"{d['ssim']:>6.4f} {d['secs']:>5.2f}")
        else:
            print(f"{d['label']:<14} {f(d['one'])} {f(d['full'])} {f(d['gain'])} | "
                  f"{f(d['psnr_vs_prior'], 8)} {f(d['p999'], 8, 4)} "
                  f"{f(d['p999'] - PRIOR[key]['p999'], 8, 4)} | "
                  f"{d['ssim']:>6.4f} {d['secs']:>5.2f}")
    print("-" * len(hd))
    p = PRIOR[key]
    if key == "tumour":
        print(f"{'prior (x1)':<14} {'':>7} {f(p['psnr'])} {'':>7} | {'':>8} "
              f"{f(p['et_psnr'])} {f(0.0, 10)} | {p['ssim']:>6.4f}")
    else:
        print(f"{'prior (x1)':<14} {'':>7} {f(p['psnr'])} {'':>7} | {f(0.0, 8)} "
              f"{f(p['p999'], 8, 4)} {f(0.0, 8, 4)} | {p['ssim']:>6.4f}")

if "tumour" in rows:
    b = max(rows["tumour"], key=lambda d: d["full_et"] if np.isfinite(d["full_et"]) else -1e9)
    print(f"\nbest ET-PSNR (with tumour) : {b['label']} ({b['full_et']:.3f} dB, "
          f"{b['et_vs_prior']:+.3f} over prior)")
if "clear" in rows:
    b = min(rows["clear"], key=lambda d: d["p999"] if np.isfinite(d["p999"]) else 1e9)
    print(f"least hallucination (clear): {b['label']} (p99.9+ = {b['p999']:.4f} vs "
          f"prior {PRIOR['clear']['p999']:.4f})")
print(f"\nNFE={NFE}, {N_PER_COHORT} slices per cohort -- see the caveats before ranking methods "
      f"that finish close together.")

## 6. Cohort contrast

Left: does the method produce enhancement where there is some? Right: does it stay quiet where
there is none? A method in the top-left of both panels is the one you want; most will trade.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
lbls = [d["label"] for d in rows[COHORT_ORDER[0]]]
xs = np.arange(len(lbls))

if "tumour" in rows:
    ax[0].plot(xs, [d["one_et"] for d in rows["tumour"]], "-o", label="one-shot")
    ax[0].plot(xs, [d["full_et"] for d in rows["tumour"]], "-s", label=f"full (nfe={NFE})")
    ax[0].axhline(PRIOR["tumour"]["et_psnr"], color="k", ls="--", lw=1, label="prior only")
    ax[0].set_ylabel("ET-PSNR (dB)"); ax[0].set_title("with tumour: does it synthesize?")

if "clear" in rows:
    ax[1].plot(xs, [d["p999"] for d in rows["clear"]], "-s", color="tab:red",
               label=f"full (nfe={NFE})")
    ax[1].plot(xs, [pos_p999(COHORTS["clear"]["x0"], results[l]["clear"]["oneshot"],
                             COHORTS["clear"]["mask"]) for l in lbls], "-o", color="tab:orange",
               label="one-shot")
    ax[1].axhline(PRIOR["clear"]["p999"], color="k", ls="--", lw=1, label="prior only")
    ax[1].set_ylabel("p99.9 of positive (recon - GT)")
    ax[1].set_title("without tumour: does it hallucinate?  (lower better)")

for key, style in (("tumour", "-o"), ("clear", "-s")):
    if key in rows:
        ax[2].plot(xs, [d["full"] for d in rows[key]], style, label=NICE[key])
        ax[2].axhline(PRIOR[key]["psnr"], ls="--", lw=1,
                      color="tab:blue" if key == "tumour" else "tab:orange")
ax[2].set_ylabel("brain PSNR (dB)"); ax[2].set_title("whole brain, both cohorts")

for a in ax:
    a.set_xticks(xs); a.set_xticklabels(lbls, rotation=30, ha="right")
    a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. Does more sampling help -- or hurt?

Quality against sampling budget, per cohort. A **flat** curve means the architecture gets nothing
from the bridge and you may as well ship the one-shot. On the tumour-free cohort watch `p99.9+`
**rise** with NFE: that is the sampler progressively injecting enhancement that should not be
there, and it is invisible to any single-pass validation.

In [ ]:
if not DO_NFE_SWEEP:
    print("DO_NFE_SWEEP is off -- set it True in the knobs cell and re-run section 4.")
else:
    ncol = 1 + len(COHORT_ORDER)
    fig, ax = plt.subplots(1, ncol, figsize=(5.4 * ncol, 4))
    panels = []
    if "tumour" in COHORTS:
        panels.append(("tumour", "et_psnr", "ET-PSNR (dB)", "with tumour: enhancement recovered"))
    if "clear" in COHORTS:
        panels.append(("clear", "p999", "p99.9 of positive (recon - GT)",
                       "without tumour: invented signal (lower better)"))
    panels.append((COHORT_ORDER[0], "psnr", "brain PSNR (dB)",
                   f"{NICE[COHORT_ORDER[0]]}: whole brain"))

    for a, (key, metric, ylab, ttl) in zip(ax, panels):
        for r in runs:
            cur = results[r["label"]][key]["nfe_curve"]
            ks = sorted(cur)
            a.plot(ks, [cur[k][metric] for k in ks], "-o", ms=4, label=r["label"])
        a.axhline(PRIOR_SWEEP[key][metric], color="k", ls="--", lw=1, label="prior only")
        a.axvline(NFE, color="gray", ls=":", lw=1)
        a.set_xscale("log"); a.set_xticks(NFE_GRID)
        a.set_xticklabels([str(k) for k in NFE_GRID])
        a.set_xlabel("NFE (network evaluations)"); a.set_ylabel(ylab)
        a.set_title(ttl, fontsize=10); a.grid(alpha=.3)
    ax[0].legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 8. Images

**No metrics on the panels** -- the numbers are in section 5. These are for looking at the images.

Each cohort has its own cell below, showing `PANEL_N` subjects. **Re-run a cell to advance to the
next `PANEL_N`**; it wraps around the `N_PER_COHORT` evaluated slices. `reshuffle()` gives a fresh
random order and rewinds both cohorts to the first page, `reshuffle(seed)` a reproducible one.

You can only page through slices that were actually reconstructed, so `N_PER_COHORT` in the knobs
cell caps how many subjects exist -- raise it and re-run section 4 to see more.

Everything is on the fixed `[VMIN, VMAX]` window so columns are comparable and a faded prediction
reads as faded rather than being stretched back to full range. The lime contour is the ET mask.
`SHOW_ERRORS` adds the signed `recon - T1ce` map (`bwr`, white = 0) for the same subjects
underneath.

In [ ]:
SHOW_ERRORS = True          # also draw the signed recon - T1ce map for the same subjects

scored = [r["label"] for r in runs]
PAGE, VIEW = {}, {}


def reshuffle(seed=None):
    """Fresh random display order for both cohorts, rewound to the first page. `seed=None` uses
    fresh entropy, so calling it repeatedly keeps giving new orders; pass an int to reproduce
    one."""
    rng = np.random.default_rng(seed)
    for k in COHORT_ORDER:
        VIEW[k] = rng.permutation(COHORTS[k]["x0"].shape[0])
        PAGE[k] = 0
    print("display order reshuffled; both cohorts rewound to page 1")


def next_page(key, n=None):
    """The next `n` slice indices for this cohort, advancing the cursor and wrapping."""
    n = n or PANEL_N
    idx, p = VIEW[key], PAGE[key]
    npages = max(1, int(np.ceil(len(idx) / n)))
    sel = [int(idx[(p * n + j) % len(idx)]) for j in range(min(n, len(idx)))]
    PAGE[key] = (p + 1) % npages
    return sel, p + 1, npages


def outline(a, et_slice, color="lime", lw=0.6):
    e = npy(et_slice)
    if e.max() > 0:
        a.contour(e, levels=[0.5], colors=color, linewidths=lw)


def grid(key, sel, mode="image"):
    """One figure: rows = the selected subjects, columns = inputs / methods / GT (mode='image')
    or prior / methods (mode='error'). No metric annotations -- section 5 has the numbers."""
    co = COHORTS[key]
    if mode == "image":
        cols = [("T1 (prior)", co["x1"]), ("T2", co["T2"]), ("FLAIR", co["FLAIR"])]
        cols += [(l, results[l][key]["recon"]) for l in scored]
        cols += [("T1ce GT", co["x0"])]
        cmap, vlo, vhi = "gray", VMIN, VMAX
    else:
        cols = [("prior - GT", co["x1"] - co["x0"])]
        cols += [(l, results[l][key]["recon"] - co["x0"]) for l in scored]
        cmap, vlo, vhi = "bwr", -VMAX, VMAX

    fig, ax = plt.subplots(len(sel), len(cols), squeeze=False,
                           figsize=(1.95 * len(cols), 2.3 * len(sel)))
    im_ = None
    for row, i in enumerate(sel):
        for j, (name, img) in enumerate(cols):
            a = ax[row, j]
            im_ = a.imshow(npy(img[i, 0] * co["mask"][i, 0]), cmap=cmap, vmin=vlo, vmax=vhi)
            if row == 0:
                a.set_title(name, fontsize=8)
            if j == 0:
                a.set_ylabel(f"#{i}  ET {co['et_px'][i]} px", fontsize=7)
            outline(a, co["et"][i, 0], color="k" if mode == "error" else "lime",
                    lw=0.5 if mode == "error" else 0.6)
            a.set_xticks([]); a.set_yticks([])
    if mode == "error" and im_ is not None:
        fig.colorbar(im_, ax=ax, shrink=0.6,
                     label=f"recon - T1ce  (fixed +/-{VMAX:g}, white = 0)")
    plt.tight_layout(); plt.show()


def show(key):
    """Draw the next page for one cohort. Re-run the calling cell to advance."""
    if key not in COHORTS:
        print(f"cohort '{key}' is empty -- nothing to show")
        return
    sel, page, npages = next_page(key)
    print(f"{NICE[key]} -- page {page}/{npages}: slices {sel}   "
          f"(ET px: {[COHORTS[key]['et_px'][i] for i in sel]})")
    grid(key, sel, "image")
    if SHOW_ERRORS:
        grid(key, sel, "error")


reshuffle(SEED)             # deterministic first order; call reshuffle() for a new one

### 8a. With tumour

Re-run this cell to page to the next `PANEL_N` subjects. Look inside the lime contour: whether the
enhancement is there at all, and whether its shape and intensity match the ground truth.

In [ ]:
show("tumour")

### 8b. Without tumour

Re-run this cell to page to the next `PANEL_N` subjects. There is no contour to draw here -- these
slices have no enhancing tumour. What you are looking for is the opposite of 8a: anything a method
has *added* that the T1ce ground truth does not have. On the error maps, red is invented signal.

In [ ]:
show("clear")

## 9. Enhancing-tumour closeup

The tumour-cohort slice with the most enhancing tumour, zoomed. This is where the methods are
actually being asked to differ; at whole-slice scale the difference is a few hundred voxels and
invisible.

In [ ]:
if "tumour" not in COHORTS:
    print("no tumour cohort -- nothing to zoom into")
else:
    co = COHORTS["tumour"]
    areas = co["et"].flatten(1).sum(1)
    i = int(areas.argmax())
    ys, xs_ = torch.nonzero(co["et"][i, 0], as_tuple=True)
    pad = 24
    y0, y1 = max(0, int(ys.min()) - pad), min(co["et"].shape[-2], int(ys.max()) + pad)
    x0_, x1_ = max(0, int(xs_.min()) - pad), min(co["et"].shape[-1], int(xs_.max()) + pad)
    sl = (slice(y0, y1), slice(x0_, x1_))

    zc = [("T1 (prior)", co["x1"])]
    zc += [(r["label"], results[r["label"]]["tumour"]["recon"]) for r in runs]
    zc += [("T1ce GT", co["x0"])]
    fig, ax = plt.subplots(1, len(zc), figsize=(2.3 * len(zc), 2.9), squeeze=False)
    for j, (name, img) in enumerate(zc):
        a = ax[0, j]
        a.imshow(npy(img[i, 0])[sl], cmap="gray", vmin=VMIN, vmax=VMAX)
        e = npy(co["et"][i, 0])[sl]
        if e.max() > 0:
            a.contour(e, levels=[0.5], colors="lime", linewidths=0.8)
        ttl = name
        a.set_title(ttl, fontsize=8); a.set_xticks([]); a.set_yticks([])
    plt.suptitle(f"cohort slice {i}: ET = {int(areas[i])} px "
                 f"({float(co['et'][i].mean()):.4%} of the crop)", fontsize=10)
    plt.tight_layout(); plt.show()

## Caveats

* **Cohort size.** Every number in section 5 is over `N_PER_COHORT` slices per cohort.
  Differences of a few tenths of a dB are noise at the default 16 -- raise it (and re-run
  section 4) before ranking methods that finish close together. `p99.9+` is a tail statistic and
  needs more slices than PSNR does to settle.
* **The image cells page over the same pool.** They show subjects that were already
  reconstructed, so they cost nothing extra -- but they also cannot show you a subject outside
  `N_PER_COHORT`. If four pages all look alike, widen the pool rather than re-running the cell.
* **Three of these methods see three contrasts and two see one.** UNet (all), SBCDLNet and
  SBGroupCDL get FLAIR/T1/T2; UNet (T1) and CDLNet (T1) get only the bridge state. Section 1
  groups them. A gap across groups is partly an information gap, not purely an architecture
  result -- and the tumour-free cohort is where that matters most, since FLAIR and T2 are exactly
  what tells you a lesion is *not* enhancing.
* **`p99.9+` is a proxy, not a hallucination detector.** It catches bright invented signal
  anywhere in the brain, including legitimate vessel enhancement that T1ce genuinely has and T1
  does not. Read it next to the tumour-free error maps rather than on its own.
* **PSNR ranks the wrong thing if you care about realism.** A stochastic bridge trades distortion
  for perceptual quality, so the best-PSNR method is often the one behaving most like a
  regressor. If the question is whether the enhancement *looks* real, add LPIPS or sample
  repeatedly and look at the spread.
* **No EMA.** No training loop writes `ema.pt`, so these are raw final weights. It applies
  equally to all five, but it is not the number a paper would report for the UNet.